In [ ]:
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns
import polars as pl

import lightgbm as lgb
from sklearn.model_selection import train_test_split, KFold
from sklearn.metrics import roc_auc_score

import joblib

import os

import numpy as np
from pathlib import Path

In [ ]:
path_train_x = "data\\train\\res_x.parquet"
path_train_y = "data\\train\\res_y.parquet"
path_test    = "data\\test\\res.parquet"
path_summit  = "data\\submission\\"
path_sample_submit = "data\\sample_submit.parquet"

path_save_final_model = "data\\model\\final_model"

In [ ]:
train_x = pd.read_parquet(path_train_x)
train_y = pd.read_parquet(path_train_y)
test = pd.read_parquet(path_test)
sample_submission = pd.read_parquet(path_sample_submit)

In [ ]:
num_cols    = list(train_x.columns[train_x.columns.str.startswith("num")])
cat_cols    = list(train_x.columns[train_x.columns.str.startswith("cat")])
target_cols = list(train_y.columns[train_y.columns.str.startswith("target")])

In [ ]:
params = {
    "objective": "binary",
    "metric": "auc",
    "boosting_type": "gbdt",
    "learning_rate": 0.05,
    "num_leaves": 63,
    "max_depth": -1,
    "min_child_samples": 50,
    "subsample": 0.8,
    "subsample_freq": 1,
    "colsample_bytree": 0.8,
    "reg_alpha": 0.0,
    "reg_lambda": 0.0,
    "n_estimators": 3000,
    "random_state": 42,
    "n_jobs": -1,
    "verbosity": -1,
}

In [ ]:
def loading_unloading_wrapper(path_model, code_study_f, params, train_y):
    print(f"{train_y.name}")
    if not os.path.exists(path_model):
        model = code_study_f(train_y, **params)
        joblib.dump(model, path_model)
    else:
        model = joblib.load(path_model)

    return model

In [ ]:
def study_model_all_feature(train_y, **params):
    model = lgb.LGBMClassifier(**params)

    model.fit(train_x,
              train_y,
              eval_metric="auc",
              categorical_feature=cat_cols,
              callbacks=[
                  lgb.early_stopping(stopping_rounds=100, verbose=False),
                  lgb.log_evaluation(period=100),
              ],
              )

    return model

In [ ]:
predicts = pd.DataFrame(columns = sample_submission.columns)
predicts["customer_id"] = sample_submission["customer_id"]

predict_cols = [col for col in sample_submission.columns if col != "customer_id"]

for target_col, predict_col in zip(target_cols, predict_cols):
    train_y_ = train_y[target_col]
    model = loading_unloading_wrapper(Path(f"{path_save_final_model}{target_col}.pkl"), study_model_all_feature, params, train_y_)
    predict = model.predict_proba(test)[:, 1]
    predicts[predict_col] = predict

predicts.write_parquet(f"{path_summit}\\submit1.parquet")